# Cross-Species Fruit Quality Grading via Few-Shot Prototypical Networks

**Author:** Amr Samir  
**Master's Thesis – 2026**

---

**Research Gap:** While models can classify fruit species, they fail to generalize quality grading (Good/Bad) across unseen fruit types.  
**Key Contribution:** Train on {Apple, Banana, Grape} → Test on {Mango, Orange} WITHOUT retraining.

In [ ]:
import os, sys, random, warnings
import numpy as np
import torch
warnings.filterwarnings('ignore')

# Ensure the repo root is on the path so `src` and `config` are importable
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from config import Config, ensure_dirs
from src.transforms import build_transforms
from src.dataset import FruitQualityDataset, EpisodicDataLoader
from src.models import PrototypicalNetwork
from src.losses import PrototypicalLoss
from src.train import train_protonet, evaluate, load_checkpoint
from src.visualization import (
    plot_training_history, visualize_embeddings,
    plot_confusion_matrices, plot_ablation_nshot,
)
from src.experiments import (
    test_on_unseen_fruits, ablation_n_shot,
    run_all_fsl_baselines, backbone_ablation,
    run_single_species_baselines,
    species_split_cross_validation,
    statistical_significance_tests,
    component_ablation,
    collect_dataset_statistics,
    generate_thesis_summary, generate_latex_tables,
)

# ── Reproducibility ──
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

config = Config()
ensure_dirs(config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 1 · Dataset Statistics

In [ ]:
dataset_stats = collect_dataset_statistics(
    config.DATA_ROOT, config.TRAIN_FRUITS, config.TEST_FRUITS, config.CLASSES
)

## 2 · Build Transforms & Datasets

In [ ]:
transforms_dict = build_transforms(config)
train_transform   = transforms_dict['train']
support_transform = transforms_dict['support']
eval_transform    = transforms_dict['eval']

# Training & validation (seen fruits)
train_dataset = FruitQualityDataset(
    config.DATA_ROOT, config.TRAIN_FRUITS,
    support_transform=support_transform, query_transform=train_transform,
    split='train', val_ratio=config.VAL_SPLIT_RATIO, seed=42,
)
val_dataset = FruitQualityDataset(
    config.DATA_ROOT, config.TRAIN_FRUITS,
    support_transform=eval_transform, query_transform=eval_transform,
    split='val', val_ratio=config.VAL_SPLIT_RATIO, seed=42,
)

# Test (unseen fruits)
test_dataset = FruitQualityDataset(
    config.DATA_ROOT, config.TEST_FRUITS,
    support_transform=eval_transform, query_transform=eval_transform,
    split='all',
)

print(f'Train fruits: {config.TRAIN_FRUITS}')
print(f'Test  fruits: {config.TEST_FRUITS}')

## 3 · Build Model & Train

In [ ]:
model = PrototypicalNetwork(
    backbone=config.BACKBONE,
    embedding_dim=config.EMBEDDING_DIM,
    pretrained=config.PRETRAINED,
    dropout_rate=config.DROPOUT_RATE,
    temperature=config.TEMPERATURE,
).to(device)

criterion = PrototypicalLoss(
    label_smoothing=config.LABEL_SMOOTHING,
    contrastive_weight=config.CONTRASTIVE_WEIGHT,
    temperature=config.TEMPERATURE,
)

train_loader = EpisodicDataLoader(
    train_dataset, config.N_SHOT, config.N_QUERY, config.N_EPISODES_TRAIN,
)
val_loader = EpisodicDataLoader(
    val_dataset, config.N_SHOT, config.N_QUERY, config.N_EPISODES_VAL,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {trainable:,} trainable / {total:,} total')

In [ ]:
history = train_protonet(model, train_loader, val_loader, criterion, config, device)

In [ ]:
plot_training_history(history, config)

## 4 · Evaluate on Unseen Fruits

In [ ]:
main_results = test_on_unseen_fruits(model, test_dataset, device, config, n_trials=5)

## 5 · Embedding Visualisation (t-SNE)

In [ ]:
emb2d, labels, fruits = visualize_embeddings(
    model, test_dataset, eval_transform, device, config
)

## 6 · Confusion Matrices

In [ ]:
cm = plot_confusion_matrices(model, test_dataset, device, config)

## 7 · N-Shot Ablation Study

In [ ]:
nshot_results = ablation_n_shot(model, test_dataset, device, config)
plot_ablation_nshot(nshot_results, config)

## 8 · Baseline Comparisons

In [ ]:
baseline_results = run_all_fsl_baselines(
    model, train_dataset, val_dataset, test_dataset, config, device
)

## 9 · Statistical Significance Tests

In [ ]:
sig_table = statistical_significance_tests(baseline_results)

## 10 · Backbone Ablation

In [ ]:
backbone_results = backbone_ablation(
    train_dataset, val_dataset, test_dataset, config, device
)

## 11 · Single-Species Baselines

In [ ]:
single_sp_results = run_single_species_baselines(
    model, test_dataset, transforms_dict, config, device
)

## 12 · Species-Split Cross-Validation

In [ ]:
cv_results = species_split_cross_validation(transforms_dict, config, device)

## 13 · Component Ablation Study

In [ ]:
ablation_results = component_ablation(
    model, train_dataset, val_dataset, test_dataset, config, device
)

## 14 · Thesis Summary & LaTeX Tables

In [ ]:
all_results = {
    'main_experiment': main_results,
    'ablation_nshot':  nshot_results,
    'baselines':       baseline_results,
}

generate_thesis_summary(all_results, dataset_stats, config)
generate_latex_tables(all_results, config)